# Tutorial 2.1 - Stochastic Treatment of Genetic Information Processing Reactions (Transcription, Translation, RNA Degradation and Protein Degradation) #
## Introduction ##
We will now stochastically model a well-mixed biological system! In this genetic information processing model of an arbitrary gene, we include the transcription, translation, and degradation of both mRNA and protein products. We hope to demonstrate that modeling reduced biochemical systems is simply an extension of the methods used in tutorial 1.2.

## Set the Working Directory and Import Packages ##

In [ ]:
# Import necessary packages #
import os, sys

# Define function to find desired working directory #
def _find_tutorial_dir(notebook_filename):
    """Locate this notebook's directory by searching upward from CWD."""
    start = os.path.abspath(os.getcwd())
    search = start
    for _ in range(8):
        if os.path.isfile(os.path.join(search, notebook_filename)):
            return search
        for subdir in ['LM/CME/Tutorial02_GeneticInformationProcessing',
                       'CME/Tutorial02_GeneticInformationProcessing',
                       'Tutorial02_GeneticInformationProcessing']:
            candidate = os.path.abspath(os.path.join(search, subdir))
            if os.path.isfile(os.path.join(candidate, notebook_filename)):
                return candidate
        search = os.path.dirname(search)
    return None

# Define desired working directory #
_here = _find_tutorial_dir('Tut.2.1-GeneticInformationProcessing.ipynb')
if _here is None:
    raise RuntimeError(
        "Cannot locate Tutorial02_GeneticInformationProcessing/. "
        "Please ensure the repository structure is intact."
    )

# Set working directory #
os.chdir(_here)
print(f"Working directory set to:\n{_here}")

In [ ]:
# Import Standard Python Libraries #
import numpy as np

# Import jLM Libraries #
import jLM.CME as CME
import jLM.units as units
import jLM.CMEPostProcessing as PostProcessing

# Enable plotting inline in the Jupyter notebook #
%matplotlib inline

# Add custom plotting script #
sys.path.insert(0, os.path.abspath('../Utils'))
import custom_plot as plot

## Define the CME Simulation Object ##
As was done in tutorial 1.2, we will now define an empty CME simulation object and name the simulation "Gene Expression".

In [ ]:
# Create our CME simulation object #
sim = CME.CMESimulation(name='Gene Expression')

## Define the Biological System ##
Because we will only be using 1st-order reactions in this system, we do not need to specify the volume of the system or Avogadro's number. Rather, we simply need to specify the total amount of biological time we will run the simulation, which we will set to the length of the minimal cell's cell cycle.

In [ ]:
# Define our chemical species #
species = ['gene', 'mRNA', 'protein']
# Add chemical species to the simulation objhect #
sim.defineSpecies(species)

# Set our initial species counts #
sim.addParticles(species='gene', count=1)
sim.addParticles(species='mRNA', count=1)
sim.addParticles(species='protein', count=148)

Next, we will define our rate constants and reaction network.

In [ ]:
# Constants
k_transcription  = 6.41e-4       # Transcription, s^-1
k_degra_mRNA = 2.59e-3     # degradation of mRNA, s^-1
k_translation = 7.2e-2        # translation, s^-1
k_degra_ptn = 7.70e-6      # degradation of protein, s^-1

# Add reactions to the simulation
sim.addReaction(reactant='gene', product=('gene','mRNA'), rate=k_transcription)
sim.addReaction(reactant='mRNA', product='', rate=k_degra_mRNA)
sim.addReaction(reactant='mRNA', product=('mRNA','protein'), rate=k_translation)
sim.addReaction(reactant='protein', product='', rate=k_degra_ptn)


Finally, we define the simulation execution parameters. We will have the simulation run for 6,300 seconds of biological time to cover the entire cell cyle. We will also set the write interval (frequency with which information about the state of the system is saved) to be 1 second. Then we name the simulation output file and save the simulation definition to the named file.

In [ ]:
# Set simulation parameters #
# Define simulation time as 6300s (entire cell cycle of the minimal cell) #
simtime = 6300
# Set simulation time #
sim.setSimulationTime(simtime)
# Define the write interval as 1 second #
writeInterval = 1
# Set simulation write interval #
sim.setWriteInterval(writeInterval)
# Define the number of replicates to perform of the simulation #
reps = 10
# Set simulation file output name #
filename = "./T2.1-GeneticInformationProcess.lm"
# Remove previously generated output files with the same name #
os.system("rm -rf %s"%(filename))
# Save the simulation #
sim.save(filename)

Finally, before performing the actual simulations, we can optionally print the simulation object information to make sure we have entered everything correctly.

In [ ]:
# Print simulation parameters to the notebook for checking before final sim.run #
print(f"Name:           {sim.name}")
print(f"Species:        {sim.species_id}")
print(f"Initial counts: {sim.initial_counts}")
print(f"Parameters:     {sim.parameters}")
print("Reactions:")
for reactant, product, rate in sim.reactions:
    print(f"  {reactant} -> {product}  (rate={rate})")

## Run the CME Simulation ##
As with tutorial 1.2, we will now perform the simulation using Lattice Microbes.

In [ ]:
sim.run(filename=filename, method="lm::cme::GillespieDSolver", replicates=reps)

## Analysis of Simulation Data ##

Congratulations! You have run your first (biological) LM simulation! 

As we did in tutorial 1.2, we will now use our custom plotting script to extract and plot trajectory information from our biological simulation. First, we must create the plot output folder if it is not already created.

In [ ]:
# Create folder to store plotted figures
fig_dir = './Plots/'
# Ensure the output plot directory exists if not already created #
if not os.path.exists(fig_dir):
    os.mkdir(fig_dir)

Next we extract the trajectories in LM file to a 3D Numpy Array with dimesions *(time, species, replicates)*.

In [ ]:
# Define the file handle for the LM output file generated by simulations #
fileHandle = PostProcessing.openLMFile(filename)
# Use PostProcessing to get the simulation timesteps #
timestep = PostProcessing.getTimesteps(fileHandle) 

# Initialize 3D numpy array full of zeros with proper dimensions to store trajectory data #
trajectory_temp = np.zeros((len(timestep), len(sim.particleMap), reps)) 

# Use custome plotting function in the /Utils/ directory to read the trajectories for each species and each replicate into the 3D numpy array #
trajectories = plot.get_sim_data(trajectory_temp, reps, filename)
# Check that the species trajectories were loaded in correctly #
print('The size of the 3D trajectories is {0} with dimensions time, species, and replicates.'.format(np.shape(trajectory_temp)))

Now that we have extracted the data, we will plot average abundance of both mRNA and protein species across all 10 replicates. These will be shown with solid lines. The minimum and maximum counts of each species will be shown in the shaded areas.

In [ ]:
# Define mRNA and protein trajectories individually #
trace_mRNA = trajectories[:,1,:] # 2D array
trace_ptn = trajectories[:,2,:] # 2D array

# Convert time steps to minutes #
time = timestep/60
# Define x-axis label #
xlabel = 'Time [Min]'
# Define plot title #
title = f'Trajectories of DnaA mRNA and\nProtein Across {reps} Replicates'
# Define percentile of data to plot with shaded area #
percentile = [0,100] # Full span
# Set figure width by height in inches #
fig_size = [10, 7]
# Define figure name #
fig_name = f'GIP_mRNA_Protein_{reps}Replicates'

# Define mRNA data #
left_data = [trace_mRNA]
# Define mRNA color #
left_colors = ['red']
# Define y-axis label (left)
left_ylabel = f'mRNA'
# Define mRNA plot #
left_plots = ['range_avg']
# Define y-axis label color for mRNA # 
left_ylabel_color = 'red'
# Define mRNA legends #
left_legends = len(left_data)*['']

# Define protein data #
right_data = [trace_ptn]
# Define protein color #
right_colors = ['blue']
# Define y-axis label (right) #
right_ylabel = f'Protein'
# Define protein plot #
right_plots = ['range_avg']
# Define y-axis label color for protein
right_ylabel_color = 'blue'
# Define protein legends #
right_legends = len(left_data)*['']

# Plot the figure #
plot.plot_time_dualAxes(fig_dir, fig_name, fig_size,
            time, xlabel, title, percentile,
            left_data, left_legends, left_colors, left_ylabel, left_plots, left_ylabel_color,
            right_data, right_legends, right_colors, right_ylabel, right_plots, right_ylabel_color,
            xlimit=[0,simtime/60], title_set=True, fonts_sizes=[21, 21, 24, 18],
            extension='.png', tick_setting=[12, 4.5, 15, 'out'], line_widths = [3, 4.5], legend_pos='best')

Next, we will visualize how mRNA and protein abundances change across time within a single replicate (cell). Generally, you will see an increase/burst in protein count when there are mRNAs and a degradation of protein when no mRNA is present.

Optional: Change **`rep`** to see different pattern of stochastic protein synthesis along the cell cycle.

In [ ]:
# Define replicate to plot individually #
rep = 1
# Define mRNA and protein trajectories individually #
trace_mRNA = trajectories[:,1,:] # 2D array
trace_ptn = trajectories[:,2,:] # 2D array

# Convert time steps to minutes #
time = timestep/60
# Define x-axis label #
xlabel = 'Time [Min]'
# Define plot title #
title = f'Trajectories of DnaA mRNA and\nProtein in Replicate {rep}'
# Define percentile of data to plot with shaded area #
percentile = [0,100] # Full span
# Set figure width by height in inches #
fig_size = [10, 7]
# Define figure name #
fig_name = f'GIP_mRNA_Protein_Cell{rep}'

# Define mRNA data #
left_data = [trace_mRNA[:,rep-1]]
# Define mRNA color #
left_colors = ['red']
# Define y-axis label (left)
left_ylabel = f'mRNA'
# Define mRNA plot #
left_plots = ['single']
# Define y-axis label color for mRNA # 
left_ylabel_color = 'red'
# Define mRNA legends #
left_legends = len(left_data)*['']

# Define protein data #
right_data = [trace_ptn[:,rep-1]]
# Define protein color #
right_colors = ['blue']
# Define y-axis label (right) #
right_ylabel = f'Protein'
# Define protein plot #
right_plots = ['single']
# Define y-axis label color for protein
right_ylabel_color = 'blue'
# Define protein legends #
right_legends = len(left_data)*['']

# Plot the figure #
plot.plot_time_dualAxes(fig_dir, fig_name, fig_size,
            time, xlabel, title, percentile,
            left_data, left_legends, left_colors, left_ylabel, left_plots, left_ylabel_color,
            right_data, right_legends, right_colors, right_ylabel, right_plots, right_ylabel_color,
            xlimit=[0,simtime/60], title_set=True, fonts_sizes=[21, 21, 24, 18],
            extension='.png', tick_setting=[12, 4.5, 15, 'out'], line_widths = [3, 4.5], legend_pos='best')

Finally, we will visualize the amount of protein that has been generated by he end of the cell cycle for all 10 replicates (cells). Notice that there is a considerable spread in abundances, which will only be captured using stochastic simulation techniques.

In [ ]:
# Define protein abundance at the end of the cell cycle #
ptn_endcycle = trace_ptn[-1,:] # 1D array

# Set figure width by height in inches #
fig_size = [7, 7]
# Define figure name #
fig_name = f'GIP_Proteins_CycleEnd_{reps}replicates'
# Define data to plot #
data_list = [ptn_endcycle]
# Define legend #
legends = ['']
# Define color #
colors = ['blue']
# Define x-axis label #
xlabel ='Protein Counts at Cycle End'
# Define y-axis label #
ylabel ='Cells'
# Define plot title #
title = f'DnaA Protein Distribution\nAcross {reps} Replicates'
# Define number of histogram bins #
bins = 10

# Plot the figure #
plot.plot_hists(fig_dir, fig_name, fig_size,
            data_list, legends, colors, xlabel, ylabel, title, bins,
            mean_median=[False, False],
            title_set=True, fonts_sizes=[21, 21, 21, 18],
            extension='.png', range=None, 
            tick_setting=[12, 4.5, 18, 'out'], line_widths = [3, 4.5], legend_pos='upper left')